## NVIDIA News (AI) Scraper

In [1]:
# NVIDIA News Scraper
try:
    from curl_cffi import requests
except ImportError:
    print("Error: curl_cffi not installed. Please run: pip install curl-cffi")
    import requests

from bs4 import BeautifulSoup
from datetime import datetime, timedelta
import time
import re

# ==========================================
# NVIDIA News Scraper Helpers
# ==========================================

def parse_nvidia_date(date_str):
    """Parse dates like 'December 22, 2025' or 'Dec 22, 2025'"""
    if not date_str: return None
    date_str = date_str.strip()
    formats = [
        '%B %d, %Y',       # December 22, 2025
        '%b %d, %Y',       # Dec 22, 2025
        '%Y-%m-%d',
    ]
    for fmt in formats:
        try:
            return datetime.strptime(date_str, fmt)
        except ValueError:
            continue
    return None

def normalize_date(d):
    if isinstance(d, str):
        for fmt in ['%Y-%m-%d', '%B %d, %Y', '%b %d, %Y']:
             try:
                 return datetime.strptime(d, fmt)
             except:
                 continue
    return d

def get_nvidia_news_list(start_date, end_date=None):
    """
    Scrapes article list from https://nvidianews.nvidia.com/news?q=ai
    """
    url = "https://nvidianews.nvidia.com/news?q=ai&year="
    
    try:
        start_dt = normalize_date(start_date)
        end_dt = normalize_date(end_date) if end_date else datetime.now()
        if start_dt: start_dt = start_dt.replace(hour=0, minute=0, second=0)
        if end_dt: end_dt = end_dt.replace(hour=23, minute=59, second=59)
        
        print(f"Fetching NVIDIA News List... Range: {start_dt} ~ {end_dt}")
        
        response = requests.get(url, impersonate="chrome", timeout=30)
        if response.status_code != 200:
             print(f"Failed to access list page: {response.status_code}")
             return []
            
        soup = BeautifulSoup(response.content, 'html.parser')
        articles = []
        
        # Selectors:
        # Card: div.index-item
        # Title: .index-item-text-title a
        # Date: .index-item-text-info-date or scan text
        
        items = soup.find_all('div', class_=lambda c: c and 'index-item' in c)
        
        for item in items:
            try:
                # Title & URL
                title_el = item.find(class_=lambda c: c and 'index-item-text-title' in c)
                link = None
                if title_el: link = title_el.find('a')
                
                if not link:
                    # Fallback
                    link = item.find('a', href=True)
                
                if not link: continue
                
                href = link.get('href')
                full_url = href if href.startswith('http') else f"https://nvidianews.nvidia.com{href}"
                title = link.get_text(strip=True)
                
                # Date Extraction
                # Try specific class first
                date_str = ""
                found_date = None
                
                date_el = item.find(class_=lambda c: c and 'date' in c)
                if date_el:
                     date_str = date_el.get_text(strip=True)
                else:
                     # Heuristic: search text for date pattern
                     text = item.get_text(" ")
                     match = re.search(r'([A-Z][a-z]+ \d{1,2}, \d{4})', text)
                     if match: date_str = match.group(1)
                
                found_date = parse_nvidia_date(date_str)
                
                if found_date:
                    if start_dt <= found_date <= end_dt:
                         if not any(a['url'] == full_url for a in articles):
                             articles.append({
                                 'url': full_url,
                                 'date': date_str,
                                 'dt': found_date,
                                 'title': title
                             })
            except Exception as e:
                print(f"Error parsing item: {e}")
                continue

        print(f"Found {len(articles)} matching articles in list.")
        return articles
        
    except Exception as e:
        print(f"Error in get_nvidia_news_list: {e}")
        return []

def scrape_nvidia_article_detail(url, list_info):
    """
    Scrapes detail of a single NVIDIA article.
    """
    try:
        response = requests.get(url, impersonate="chrome", timeout=30)
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Main Content
        # .entry-content
        content_div = soup.find(class_=lambda c: c and 'entry-content' in c)
        
        if not content_div:
             content_div = soup.find('article')
        
        if content_div:
            for tag in content_div(['script', 'style', 'button', 'nav', 'footer', 'header', 'form', 'iframe']):
                tag.decompose()
            raw_content = content_div.get_text(separator='\n', strip=True)
        else:
            raw_content = ""
            
        # Title extraction from Page
        h1 = soup.find(class_=lambda c: c and 'entry-title' in c)
        if not h1: h1 = soup.find('h1')
        page_title = h1.get_text(strip=True) if h1 else list_info['title']
        
        return {
            "raw_content": raw_content,
            "source_url": url,
            "date": list_info['date'],
            "raw_title": page_title,
            "source_name": "NVIDIA News"
        }

    except Exception as e:
        print(f"Error scraping article {url}: {e}")
        return None

def run_nvidia_scraper(start_date, end_date=None):
    target_articles = get_nvidia_news_list(start_date, end_date)
    results = []
    
    for article in target_articles:
        print(f"  > Scraping: {article['title']} ({article['date']})")
        details = scrape_nvidia_article_detail(article['url'], article)
        if details:
             results.append(details)
        time.sleep(1)
        
    print(f"Done. Scraped {len(results)} articles.")
    return results


In [2]:
# Test NVIDIA Scraper
if 'run_nvidia_scraper' in globals():
    # Example Range
    # Set a relevant range for expected news
    start_date = '2025-12-20'
    end_date = '2026-01-31'
    
    print(f"Running scraper from {start_date} to {end_date}...")
    res = run_nvidia_scraper(start_date, end_date)
    
    print("\n[Results]")
    for r in res:
        print(r)


Running scraper from 2025-12-20 to 2026-01-31...
Fetching NVIDIA News List... Range: 2025-12-20 00:00:00 ~ 2026-01-31 23:59:59
Found 1 matching articles in list.
  > Scraping: Marine Biological Laboratory Explores Human Memory With AI and Virtual Reality (December 22, 2025)
Done. Scraped 1 articles.

[Results]
{'raw_content': 'The works of Plato state that when humans have an experience, some level of change occurs in their brain, which is powered by memory — specifically long-term memory.\nThis change is what Andre Fenton, professor of neural science at New York University, and Abhishek Kumar, assistant professor of cell and regenerative biology at the University of Wisconsin–Madison, are studying at the Marine Biological Laboratory (MBL) in Woods Hole, Massachusetts.\n“My life’s work is to understand how minds operate, and especially to understand memory — not merely as a trace of the past in the brain but as an estimate of the future that the brain is afforded,” Fenton said.\nThe re

In [3]:
r

{'raw_content': 'The works of Plato state that when humans have an experience, some level of change occurs in their brain, which is powered by memory — specifically long-term memory.\nThis change is what Andre Fenton, professor of neural science at New York University, and Abhishek Kumar, assistant professor of cell and regenerative biology at the University of Wisconsin–Madison, are studying at the Marine Biological Laboratory (MBL) in Woods Hole, Massachusetts.\n“My life’s work is to understand how minds operate, and especially to understand memory — not merely as a trace of the past in the brain but as an estimate of the future that the brain is afforded,” Fenton said.\nThe researchers have upleveled their project by harnessing\nNVIDIA RTX GPUs\nand\nHP Z Workstations\nto visualize massive datasets and by integrating custom AI tools and\nsyGlass\n, a virtual-reality (VR) platform for scientific exploration.\nThis project is additionally supported by grants from the National Institut